# Tutorial 3 — Finding the saddle path by shooting

**Source:** `Tutorial_3.py`

**In class:** PS2 Question 2(f); about 30 minutes of programming.

Run cells from top to bottom. At each pause, predict the result before running the next cell. Plots appear below their cells; the source scripts remain the full resolution reference.


## 1. Two laws of motion

$$\dot k=Ak^\alpha-c-\delta k,\qquad
\frac{\dot c}{c}=\frac{\alpha Ak^{\alpha-1}-\delta-\rho}{\sigma}.$$

Given $k_0$, the initial consumption $c_0$ must place the economy on the stable path. We will locate it between two guesses that fail in opposite ways.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


In [ ]:
ALPHA = 1 / 3
A = 1.0
DELTA = 0.05
RHO = 0.03
SIGMA = 2.0


def f(k):
    return A * k**ALPHA


def f_prime(k):
    return ALPHA * A * k ** (ALPHA - 1)


def ngm(t, y):
    """The two laws of motion. y = (k, c)."""
    k, c = y
    return [f(k) - c - DELTA * k, c * (f_prime(k) - DELTA - RHO) / SIGMA]


## 2. Locate the steady state and draw the zero-change curves

$\dot c=0$ is vertical at $k^*$. $\dot k=0$ satisfies $c=f(k)-\delta k$. Their intersection is $(k^*,c^*)$. The peak of the latter curve occurs at golden-rule capital.


In [ ]:
k_star = (ALPHA * A / (RHO + DELTA)) ** (1 / (1 - ALPHA))
c_star = f(k_star) - DELTA * k_star
k_gr = (ALPHA * A / DELTA) ** (1 / (1 - ALPHA))

print(f"STEP 1   k*   = {k_star:.4f}")
print(f"         c*   = {c_star:.4f}")
print(f"         k_gr = {k_gr:.4f}   -- and k* < k_gr, as PS2 Q2(a) claims")


In [ ]:
k_grid = np.linspace(0.05, 1.6*k_gr, 500)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(k_grid, f(k_grid)-DELTA*k_grid, label=r'$\dot k=0$')
ax.axvline(k_star, color='C1', label=r'$\dot c=0$')
ax.plot(k_star, c_star, 'ko')
ax.set(xlabel='$k$', ylabel='$c$', xlim=(0, 1.6*k_gr))
ax.legend(frameon=False); plt.show()


## 3. Make the two failures observable

High $c_0$ uses up capital; low $c_0$ leads to overaccumulation and eventually consumption near zero. A tiny positive floor avoids integrating into an economically meaningless negative region.


In [ ]:
FLOOR = 1e-4


def capital_gone(t, y):
    return y[0] - FLOOR


def consumption_gone(t, y):
    return y[1] - FLOOR


capital_gone.terminal = True
consumption_gone.terminal = True


def integrate(k0, c0, t_end=400.0):
    return solve_ivp(
        ngm, [0.0, t_end], [k0, c0],
        events=[capital_gone, consumption_gone], dense_output=True,
        rtol=1e-11, atol=1e-13,
    )


In [ ]:
K0 = .3*k_star
for c0 in (1.10, 0.40):
    s = integrate(K0, c0)
    failure = 'capital' if s.t_events[0].size else 'consumption' if s.t_events[1].size else 'neither'
    print(f'c0={c0:.2f}: {failure} exhausted at t={s.t[-1]:.1f}')


## 4. One bisection step

The stable choice lies between $0.40$ and $1.10$. Try the midpoint. Keep the half interval bounded by opposite failures.


In [ ]:
low, high = .40, 1.10
mid = (low+high)/2
s = integrate(K0, mid)
print('midpoint', mid, 'capital event?', bool(s.t_events[0].size),
      'consumption event?', bool(s.t_events[1].size))


## 5. Repeat the classification and halving

The function below carries out the same midpoint test repeatedly. Its fallback handles guesses for which neither event occurs within the integration horizon.


In [ ]:
def runs_out_of_capital(k0, c0):
    """True if this guess is above the saddle path."""
    s = integrate(k0, c0)
    if s.t_events[0].size:
        return True
    if s.t_events[1].size:
        return False
    return s.y[0, -1] < k_star  # neither fired: fall back on which way k drifted


def shoot(k0, rounds=60):
    """Bisect on c0 until the trajectory from k0 stays on the saddle path."""
    low, high = 1e-10, f(k0)      # cannot consume more than output
    for _ in range(rounds):
        mid = 0.5 * (low + high)
        if runs_out_of_capital(k0, mid):
            high = mid
        else:
            low = mid
    return 0.5 * (low + high)


In [ ]:
for rounds in (2, 4, 8, 16, 32, 60):
    print(f'{rounds:2d} rounds: c0 = {shoot(K0, rounds=rounds):.8f}')


## 6. Draw the resulting path

Numerical error eventually nudges a trajectory off a saddle path. Show only the informative early portion; the solution is the initial consumption found by shooting.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(k_grid, f(k_grid)-DELTA*k_grid, label=r'$\dot k=0$')
ax.axvline(k_star, color='C1', label=r'$\dot c=0$')
for multiple in (.3, 1.8):
    k0 = multiple*k_star
    c0 = shoot(k0)
    s = integrate(k0, c0)
    points = s.sol(np.linspace(0, min(s.t[-1], 120), 500))
    ax.plot(points[0], points[1], label=rf'$k_0={multiple}k^*$')
    print(f'k0={k0:.3f} -> c0={c0:.6f}')
ax.set(xlabel='$k$', ylabel='$c$', xlim=(0, 1.6*k_gr), ylim=(0, 1.5*c_star))
ax.legend(frameon=False); plt.show()


**Pause and connect to PS2 2(f).** Halve initial capital, then raise $\rho$ from $0.03$ to $0.06$. For the second experiment, pass parameters into the model and shooting functions so the new steady state and the dynamics use the same $\rho$.
